# 🌾 ML-Based Creditworthiness Prediction & Subsidy Matching for Farmers
**PICT Batch E4 | AY 2025-26**  
Jyotiraditya Jadhav · Shriya Mokashi · Aryan Mutyal · Ojas Sangwai | Guide: Prof. Parag Jambhulkar

---
**Dataset:** AgriRiskFin_Dataset.csv — Kaggle (programmer3) | 4,981 rows × 17 columns

## 1. Setup & Load Data

In [ ]:
import glob, os, json, pickle, warnings
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, accuracy_score,
    precision_score, recall_score
)

warnings.filterwarnings('ignore')
np.random.seed(42)
os.makedirs('outputs', exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f8',
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})

In [ ]:
KAGGLE_FOLDER = r'C:\Users\shriy\.cache\kagglehub\datasets\programmer3\agriculture-financial-risk-dataset\versions\1'
CSV_PATH = glob.glob(os.path.join(KAGGLE_FOLDER, '*.csv'))[0]

df_raw = pd.read_csv(CSV_PATH)
print(f'Dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

## 2. Data Inspection & Mapping

**Data sources per SRS §11 Step 1:**
- **Farmer-provided:** Enterprise Size, Region, Revenue, Expenses, Loan Amount, Net Profit, Debt-to-Equity, Quarter
- **Auto-fetched from APIs:** Avg Temperature & Rainfall (IMD), Drought Index & Flood Risk (NDMA), Commodity & Input Cost Index (AGMARKNET), Policy Support Score (Govt DB)

**Target:** `Financial_Risk_Level` → High = Default (1), Low/Medium = Non-Default (0)

In [ ]:
# Check for nulls
print('Null values:', df_raw.isnull().sum().sum())
print('\nColumn types:')
print(df_raw.dtypes.to_string())

In [ ]:
# Map to working dataframe
df = pd.DataFrame({
    # Farmer-provided
    'enterprise_id':   df_raw['Enterprise_ID'],
    'region':          df_raw['Region'].str.strip(),
    'enterprise_size': df_raw['Enterprise_Size'].str.strip(),
    'quarter':         df_raw['Quarter'].str.strip(),
    'annual_revenue':  df_raw['Revenue'],
    'annual_expenses': df_raw['Expenses'],
    'loan_amount':     df_raw['Loan_Amount'],
    'net_profit':      df_raw['Net_Profit'],
    'DE_ratio_raw':    df_raw['Debt_to_Equity'],
    # Auto-fetched from APIs
    'avg_temperature': df_raw['Avg_Temperature'],   # IMD API
    'rainfall':        df_raw['Rainfall'],           # IMD API
    'drought_index':   df_raw['Drought_Index'],      # NDMA API
    'flood_risk':      df_raw['Flood_Risk_Score'],   # NDMA API
    'commodity_idx':   df_raw['Commodity_Price_Index'],  # AGMARKNET
    'input_cost_idx':  df_raw['Input_Cost_Index'],       # AGMARKNET
    'policy_score':    df_raw['Policy_Support_Score'],   # Govt Policy DB
    # Target
    'default':         (df_raw['Financial_Risk_Level'].str.strip() == 'High').astype(int),
    'risk_level_raw':  df_raw['Financial_Risk_Level'].str.strip(),
})

print('Risk Level distribution:')
print(df_raw['Financial_Risk_Level'].value_counts().to_string())
print(f'\nDefault rate (High=1): {df["default"].mean():.2%}')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
df.describe().T.round(2)

In [ ]:
# Fig 1 — Default rate by categories + key numeric distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('EDA — Key Patterns', fontsize=14, fontweight='bold')

# Default rate by enterprise size
ax = axes[0, 0]
dr = df.groupby('enterprise_size')['default'].mean().sort_values(ascending=False)
ax.bar(dr.index, dr.values * 100, color=['#e74c3c','#f39c12','#2ecc71'])
ax.set_title('Default Rate by Enterprise Size')
ax.set_ylabel('Default Rate (%)')
for i, v in enumerate(dr.values):
    ax.text(i, v*100 + 0.5, f'{v:.1%}', ha='center', fontweight='bold')

# Default rate by region
ax = axes[0, 1]
dr = df.groupby('region')['default'].mean().sort_values(ascending=False)
ax.bar(dr.index, dr.values * 100, color='#3498db')
ax.set_title('Default Rate by Region')
ax.set_ylabel('Default Rate (%)')
for i, v in enumerate(dr.values):
    ax.text(i, v*100 + 0.5, f'{v:.1%}', ha='center', fontweight='bold')

# Net profit distribution
ax = axes[0, 2]
df[df['default']==0]['net_profit'].hist(bins=40, alpha=0.6, color='#2ecc71',
                                         label='Non-Default', ax=ax, density=True)
df[df['default']==1]['net_profit'].hist(bins=40, alpha=0.6, color='#e74c3c',
                                         label='Default', ax=ax, density=True)
ax.set_title('Net Profit Distribution')
ax.set_xlabel('Net Profit (₹K)')
ax.legend()

# Loan amount distribution
ax = axes[1, 0]
df[df['default']==0]['loan_amount'].hist(bins=40, alpha=0.6, color='#2ecc71',
                                          label='Non-Default', ax=ax, density=True)
df[df['default']==1]['loan_amount'].hist(bins=40, alpha=0.6, color='#e74c3c',
                                          label='Default', ax=ax, density=True)
ax.set_title('Loan Amount Distribution')
ax.set_xlabel('Loan Amount (₹K)')
ax.legend()

# Drought index
ax = axes[1, 1]
df[df['default']==0]['drought_index'].hist(bins=30, alpha=0.6, color='#2ecc71',
                                            label='Non-Default', ax=ax, density=True)
df[df['default']==1]['drought_index'].hist(bins=30, alpha=0.6, color='#e74c3c',
                                            label='Default', ax=ax, density=True)
ax.set_title('Drought Index [NDMA API]')
ax.set_xlabel('Drought Index')
ax.legend()

# Commodity price index
ax = axes[1, 2]
df[df['default']==0]['commodity_idx'].hist(bins=30, alpha=0.6, color='#2ecc71',
                                            label='Non-Default', ax=ax, density=True)
df[df['default']==1]['commodity_idx'].hist(bins=30, alpha=0.6, color='#e74c3c',
                                            label='Default', ax=ax, density=True)
ax.set_title('Commodity Price Index [AGMARKNET]')
ax.set_xlabel('Index Value')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/fig1_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig 2 — Correlation heatmap
num_cols = ['annual_revenue','annual_expenses','loan_amount','net_profit',
            'DE_ratio_raw','avg_temperature','rainfall','drought_index',
            'flood_risk','commodity_idx','input_cost_idx','policy_score','default']

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones(len(num_cols), dtype=bool))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.3, mask=mask, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9}, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('outputs/fig2_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mean values by default class
print('=== Mean Values: Non-Default vs Default ===')
cols = ['annual_revenue','annual_expenses','loan_amount','net_profit',
        'DE_ratio_raw','drought_index','flood_risk','commodity_idx','policy_score']
pd.DataFrame({
    'Non-Default': df[df['default']==0][cols].mean().round(2),
    'Default':     df[df['default']==1][cols].mean().round(2),
    'Difference':  (df[df['default']==1][cols].mean() - df[df['default']==0][cols].mean()).round(2)
})

## 4. Feature Engineering
**SRS §11 Step 3** — derived features from raw inputs and API data.

| Feature | Formula | Source |
|---------|---------|--------|
| ERR | Expenses / Revenue | Farmer |
| DSR | Loan / Revenue | Farmer |
| NPM | Net Profit / Revenue | Farmer |
| D/E | Debt-to-Equity | Farmer |
| CRI | f(Rainfall, Drought, Flood) | IMD / NDMA API |
| TSI | f(Avg Temperature) | IMD API |
| MRI | Commodity Index / Input Cost Index | AGMARKNET API |

In [ ]:
fe  = df.copy()
EPS = 1e-6

# SRS §3.1 — Financial ratios
fe['ERR']      = (fe['annual_expenses'] / (fe['annual_revenue'] + EPS)).clip(0, 3)
fe['DSR']      = (fe['loan_amount']     / (fe['annual_revenue'] + EPS)).clip(0, 5)
fe['NPM']      = (fe['net_profit']      / (fe['annual_revenue'] + EPS)).clip(-2, 1)
fe['DE_ratio'] = fe['DE_ratio_raw'].clip(0, 5)

# SRS §3.2 — Climate features (from IMD / NDMA API)
rain_norm      = ((fe['rainfall'] - fe['rainfall'].min()) /
                  (fe['rainfall'].max() - fe['rainfall'].min() + EPS)).clip(0, 1)
di             = fe['drought_index'].clip(0, 1)
fr             = fe['flood_risk'].clip(0, 1)
fe['CRI']      = (0.40*rain_norm - 0.35*di - 0.25*fr + 0.35).clip(0, 1)
fe['TSI']      = ((fe['avg_temperature'] - 23.5).abs() / 10).clip(0, 1)

# SRS §3.3 — Market features (from AGMARKNET API)
fe['MRI']      = (fe['commodity_idx'] / (fe['input_cost_idx'] + EPS)).clip(0.3, 3.0)

# SRS §3.4 — Encodings
fe['enterprise_enc'] = fe['enterprise_size'].map({'Small':0,'Medium':1,'Large':2}).astype(int)
fe['policy_norm']    = ((fe['policy_score'] - fe['policy_score'].min()) /
                        (fe['policy_score'].max() - fe['policy_score'].min() + EPS)).clip(0, 1)
fe['quarter_enc']    = fe['quarter'].map({'Q1':0,'Q2':1,'Q3':2,'Q4':3}).astype(int)

# SRS §3.5 — Derived stability indicators
fe['financial_stability'] = (1 - fe['ERR'] / 2).clip(0, 1)
fe['leverage_risk']       = (fe['DE_ratio'] / 5).clip(0, 1)
fe['repayment_capacity']  = (fe['NPM'] / 2 + 0.5).clip(0, 1)

ENGINEERED = [
    'ERR', 'DSR', 'NPM', 'DE_ratio',
    'CRI', 'TSI', 'MRI',
    'enterprise_enc', 'policy_norm', 'quarter_enc',
    'financial_stability', 'leverage_risk', 'repayment_capacity',
]

print(f'Engineered features: {len(ENGINEERED)}')
fe[ENGINEERED].describe().T.round(3)

In [ ]:
# Fig 3 — Engineered features: class separation
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle('Engineered Features — Non-Default (green) vs Default (red)',
             fontsize=13, fontweight='bold')

plot_feats = [
    ('ERR','ERR — Expense/Revenue'), ('DSR','DSR — Debt/Revenue'),
    ('NPM','NPM — Net Profit Margin'), ('DE_ratio','D/E Ratio'),
    ('CRI','CRI — Climate Resilience [IMD]'), ('TSI','TSI — Temp Stress [IMD]'),
    ('MRI','MRI — Market Risk [AGMARKNET]'), ('financial_stability','Financial Stability'),
    ('leverage_risk','Leverage Risk'), ('repayment_capacity','Repayment Capacity'),
    ('policy_norm','Policy Support [Govt DB]'), ('enterprise_enc','Enterprise Size'),
]

for ax, (feat, title) in zip(axes.flat, plot_feats):
    d0 = fe[fe['default']==0][feat]
    d1 = fe[fe['default']==1][feat]
    ax.hist(d0, bins=35, alpha=0.6, color='#2ecc71', label='Non-Default', density=True)
    ax.hist(d1, bins=35, alpha=0.6, color='#e74c3c', label='Default',     density=True)
    ax.axvline(d0.mean(), color='#27ae60', lw=2, ls='--')
    ax.axvline(d1.mean(), color='#c0392b', lw=2, ls='--')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('outputs/fig3_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Training & Comparison
**SRS §11 Step 4** — Three candidate algorithms trained and compared. Best model selected for deployment.

In [ ]:
# Build feature matrix
RAW = ['annual_revenue','annual_expenses','loan_amount','net_profit',
       'avg_temperature','rainfall','drought_index','flood_risk',
       'commodity_idx','input_cost_idx','policy_score']

region_dummies = pd.get_dummies(fe['region'], drop_first=True, prefix='region')

X = pd.concat([fe[RAW], fe[ENGINEERED], region_dummies], axis=1).fillna(0)
y = fe['default'].astype(int)
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Features: {X.shape[1]} ({len(RAW)} raw + {len(ENGINEERED)} engineered + {region_dummies.shape[1]} region OHE)')

In [ ]:
# Train all three SRS candidate models
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. Logistic Regression
lr = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_sc, y_train)
lr_prob = lr.predict_proba(X_test_sc)[:,1]
lr_pred = lr.predict(X_test_sc)

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=8,
                              class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:,1]
rf_pred = rf.predict(X_test)

# 3. XGBoost
spw = (y_train==0).sum() / (y_train==1).sum()
xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     scale_pos_weight=spw, eval_metric='logloss',
                     random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_prob = xgb.predict_proba(X_test)[:,1]
xgb_pred = xgb.predict(X_test)

print('All models trained.')

In [ ]:
# Comparison table
results = {}
for name, pred, prob in [
    ('Logistic Regression', lr_pred, lr_prob),
    ('Random Forest',       rf_pred, rf_prob),
    ('XGBoost',             xgb_pred, xgb_prob),
]:
    cv_auc = cross_val_score(
        lr if name == 'Logistic Regression' else rf if name == 'Random Forest' else xgb,
        X_train_sc if name == 'Logistic Regression' else X_train,
        y_train, cv=cv5, scoring='roc_auc'
    )
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred), 4),
        'Recall':    round(recall_score(y_test, pred), 4),
        'F1':        round(f1_score(y_test, pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, prob), 4),
        'CV-AUC':    round(cv_auc.mean(), 4),
    }

comp_df = pd.DataFrame(results).T
print('=== MODEL COMPARISON ===')
print(comp_df.to_string())
print(f'\n✅ Best model: {comp_df["ROC-AUC"].idxmax()} (highest ROC-AUC)')

In [ ]:
# Fig 4 — Model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison — SRS §11 Step 4', fontsize=13, fontweight='bold')

# ROC curves
ax = axes[0]
for name, prob, color in [
    ('Logistic Regression', lr_prob,  '#3498db'),
    ('Random Forest',       rf_prob,  '#2ecc71'),
    ('XGBoost',             xgb_prob, '#e74c3c'),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f'{name}  AUC={roc_auc_score(y_test, prob):.4f}')
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(fontsize=9)

# Metric comparison bars
ax = axes[1]
metrics = ['Accuracy','F1','ROC-AUC']
x = np.arange(len(metrics))
w = 0.25
for i, (name, color) in enumerate([
    ('Logistic Regression','#3498db'),
    ('Random Forest','#2ecc71'),
    ('XGBoost','#e74c3c'),
]):
    ax.bar(x + i*w, [results[name][m] for m in metrics], w,
           label=name, color=color, alpha=0.85)
ax.set_xticks(x + w)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_title('Key Metrics Comparison')
ax.legend(fontsize=9)
ax.set_ylabel('Score')

plt.tight_layout()
plt.savefig('outputs/fig4_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Credit Score & Risk Categorization
**SRS §11 Steps 5–6** — XGBoost selected as final model (highest AUC + F1).

**Credit Score = (1 − P(default)) × 100**

| Score | Risk Category |
|:-----:|:-------------:|
| 80–100 | 🟢 Low Risk |
| 50–79 | 🟡 Medium Risk |
| 0–49 | 🔴 High Risk |

In [ ]:
# Detailed XGBoost evaluation
print('=== XGBoost — Final Model ===')
print(classification_report(y_test, xgb_pred, target_names=['Non-Default','Default']))

In [ ]:
# Compute credit scores on full dataset
prob_all = xgb.predict_proba(X.fillna(0))[:,1]
fe['P_default']    = prob_all.round(4)
fe['credit_score'] = ((1 - prob_all) * 100).round(2)

def assign_risk(s):
    if s >= 80:   return 'Low Risk'
    elif s >= 50: return 'Medium Risk'
    else:         return 'High Risk'

fe['risk_category'] = fe['credit_score'].apply(assign_risk)
risk_dist = fe['risk_category'].value_counts()
CATS = ['Low Risk','Medium Risk','High Risk']
COLORS = {'Low Risk':'#2ecc71','Medium Risk':'#f39c12','High Risk':'#e74c3c'}

print('Credit Score Summary:')
print(fe['credit_score'].describe().round(2).to_string())
print('\nRisk Distribution:')
for cat in CATS:
    n = risk_dist.get(cat, 0)
    print(f'  {cat:<15}: {n:,} ({n/len(fe):.1%})')

In [ ]:
# Fig 5 — Credit score dashboard
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Credit Score Dashboard — XGBoost', fontsize=13, fontweight='bold')

# Score distribution
ax = axes[0]
for cat in CATS:
    ax.hist(fe[fe['risk_category']==cat]['credit_score'],
            bins=25, color=COLORS[cat], alpha=0.85, label=cat)
ax.axvline(50, color='gray', ls='--', lw=1.5)
ax.axvline(80, color='gray', ls='--', lw=1.5)
ax.set_xlabel('Credit Score (0–100)')
ax.set_ylabel('Count')
ax.set_title('Score Distribution')
ax.legend()

# Pie chart
ax = axes[1]
vals = [risk_dist.get(c, 0) for c in CATS]
ax.pie(vals, labels=CATS, colors=[COLORS[c] for c in CATS],
       autopct='%1.1f%%', startangle=90)
ax.set_title('Risk Category Split')

# Confusion matrix
ax = axes[2]
cm = confusion_matrix(y_test, xgb_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non-Default','Default'],
            yticklabels=['Non-Default','Default'],
            annot_kws={'size':13,'weight':'bold'})
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
ax.set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig('outputs/fig5_credit_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Explainability — Feature Importance
**SRS §12.4** — XGBoost feature importance (gain) shows which features drive predictions most.

In [ ]:
# XGBoost feature importance
LABELS = {
    'annual_revenue':'Annual Revenue','annual_expenses':'Annual Expenses',
    'loan_amount':'Loan Amount','net_profit':'Net Profit',
    'avg_temperature':'Avg Temperature [IMD]','rainfall':'Rainfall [IMD]',
    'drought_index':'Drought Index [NDMA]','flood_risk':'Flood Risk [NDMA]',
    'commodity_idx':'Commodity Index [AGMARKNET]','input_cost_idx':'Input Cost Index',
    'policy_score':'Policy Score [Govt DB]',
    'ERR':'Expense/Revenue Ratio','DSR':'Debt/Revenue Ratio',
    'NPM':'Net Profit Margin','DE_ratio':'Debt-to-Equity',
    'CRI':'Climate Resilience Index','TSI':'Temp Stress Indicator',
    'MRI':'Market Risk Indicator','financial_stability':'Financial Stability',
    'leverage_risk':'Leverage Risk','repayment_capacity':'Repayment Capacity',
    'enterprise_enc':'Enterprise Size','policy_norm':'Policy Support (norm)',
    'quarter_enc':'Quarter',
}

imp = xgb.get_booster().get_score(importance_type='gain')
imp_df = pd.DataFrame({'feature': list(imp.keys()), 'gain': list(imp.values())})
imp_df['label'] = imp_df['feature'].map(lambda x: LABELS.get(x, x))
imp_df['pct']   = (imp_df['gain'] / imp_df['gain'].sum() * 100).round(2)
imp_df = imp_df.sort_values('gain', ascending=False)

# Fig 6
fig, ax = plt.subplots(figsize=(10, 8))
top15 = imp_df.head(15)
ax.barh(top15['label'][::-1], top15['pct'][::-1],
        color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top15))))
ax.set_xlabel('Feature Importance — Gain (%)')
ax.set_title('Top 15 Features — XGBoost (SRS §12.4)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/fig6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features:')
print(imp_df[['label','pct']].head(10).to_string(index=False))

In [ ]:
# Per-enterprise explanation
def explain(row_dict, top_n=5):
    contribs = []
    for _, r in imp_df.iterrows():
        val = float(row_dict.get(r['feature'], 0) or 0)
        contribs.append({'label': r['label'], 'value': round(val, 4),
                         'importance': round(r['pct'], 2)})
    return pd.DataFrame(contribs).sort_values('importance', ascending=False).head(top_n)

# Demo
for idx in [0, len(fe)//2, len(fe)-1]:
    f = fe.iloc[idx]
    s = f['credit_score']
    emoji = '🟢' if s>=80 else '🟡' if s>=50 else '🔴'
    print(f"\n{emoji} {f['enterprise_id']} | Score: {s:.1f}/100 | {f['risk_category']}")
    print(f"   Region: {f['region']} | Size: {f['enterprise_size']} | P(Default): {f['P_default']:.4f}")
    print('   Top factors:')
    for _, r in explain(f.to_dict()).iterrows():
        print(f"     {r['label']}: {r['value']:.3f}  (importance {r['importance']:.1f}%)")

## 8. Subsidy Matcher
**SRS §12.5** — Rule-based, independent from credit scoring. Checks 8 government schemes against farmer profile.

> `landholding_size`, `crop_type`, `irrigation_type` are farmer-provided via web form (SRS §11 Step 1A) — not in dataset, simulated here.

In [ ]:
SCHEMES = [
    {'id':'SS001','name':'PM Fasal Bima Yojana (PMFBY)',
     'benefits':'Up to ₹2 lakh crop loss compensation.',
     'rules':{'min_land':0.5,'max_land':None,'enterprise':None,
              'crops':None,'regions':None,'max_rev':None,'irrigation':None}},
    {'id':'SS002','name':'Kisan Credit Card (KCC)',
     'benefits':'Loan up to ₹3 lakh at 4% interest.',
     'rules':{'min_land':0.5,'max_land':10,'enterprise':['Small','Medium'],
              'crops':None,'regions':None,'max_rev':300,'irrigation':None}},
    {'id':'SS003','name':'PM Kisan Samman Nidhi (PM-KISAN)',
     'benefits':'₹6,000/year in 3 instalments.',
     'rules':{'min_land':0.5,'max_land':2,'enterprise':['Small'],
              'crops':None,'regions':None,'max_rev':200,'irrigation':None}},
    {'id':'SS004','name':'Per Drop More Crop (PDMC)',
     'benefits':'55% subsidy on drip/sprinkler installation.',
     'rules':{'min_land':1,'max_land':20,'enterprise':None,
              'crops':['Cotton','Sugarcane','Vegetables','Maize','Groundnut'],
              'regions':None,'max_rev':None,'irrigation':['Drip','Sprinkler','Rainfed']}},
    {'id':'SS005','name':'National Mission Sustainable Agri (NMSA)',
     'benefits':'Grants for climate adaptation and soil health.',
     'rules':{'min_land':1,'max_land':None,'enterprise':None,
              'crops':None,'regions':['East','West','South'],'max_rev':None,'irrigation':None}},
    {'id':'SS006','name':'Agricultural Infrastructure Fund (AIF)',
     'benefits':'Loans up to ₹2 crore at 3% subsidy.',
     'rules':{'min_land':2,'max_land':None,'enterprise':['Medium','Large'],
              'crops':None,'regions':None,'max_rev':None,'irrigation':None}},
    {'id':'SS007','name':'Soil Health Card Scheme',
     'benefits':'Free soil testing, reduces input costs 10–15%.',
     'rules':{'min_land':0.5,'max_land':None,'enterprise':None,
              'crops':None,'regions':None,'max_rev':None,'irrigation':None}},
    {'id':'SS008','name':'National Food Security Mission (NFSM)',
     'benefits':'Subsidised seeds, fertilisers, equipment.',
     'rules':{'min_land':0.5,'max_land':5,'enterprise':['Small','Medium'],
              'crops':['Wheat','Rice','Pulses','Maize'],
              'regions':['North','East'],'max_rev':None,'irrigation':None}},
]
print(f'{len(SCHEMES)} schemes loaded.')

In [ ]:
def match_subsidies(profile):
    """SRS §12.5 — rule-based eligibility check across all schemes."""
    land    = float(profile.get('landholding_size', 1.0))
    ent     = profile.get('enterprise_size', '')
    rev     = float(profile.get('annual_revenue', 0))
    crop    = profile.get('crop_type', '')
    region  = profile.get('region', '')
    irr     = profile.get('irrigation_type', '')
    matched = []

    for s in SCHEMES:
        r = s['rules']
        checks = []
        if r['min_land']:  checks.append(land >= r['min_land'])
        if r['max_land']:  checks.append(land <= r['max_land'])
        if r['enterprise']:checks.append(ent in r['enterprise'])
        if r['max_rev']:   checks.append(rev <= r['max_rev'])
        if r['crops']:     checks.append(crop in r['crops'])
        if r['regions']:   checks.append(region in r['regions'])
        if r['irrigation']:checks.append(irr in r['irrigation'])

        if checks and all(checks):
            matched.append({
                'scheme_id':   s['id'],
                'scheme_name': s['name'],
                'benefits':    s['benefits'],
                'match_score': round(sum(checks)/len(checks), 2),
            })

    return sorted(matched, key=lambda x: x['match_score'], reverse=True)

print('Subsidy matcher defined.')

In [ ]:
# Simulate farmer-provided inputs missing from dataset
np.random.seed(42)
n = len(fe)

fe['landholding_size'] = 1.0
fe.loc[fe['enterprise_size']=='Small',  'landholding_size'] = np.random.choice(
    [0.5,1.0,1.5,2.0], size=(fe['enterprise_size']=='Small').sum(),  p=[0.25,0.35,0.25,0.15])
fe.loc[fe['enterprise_size']=='Medium', 'landholding_size'] = np.random.choice(
    [2.0,3.0,5.0,8.0], size=(fe['enterprise_size']=='Medium').sum(), p=[0.25,0.30,0.30,0.15])
fe.loc[fe['enterprise_size']=='Large',  'landholding_size'] = np.random.choice(
    [8.0,12.0,20.0], size=(fe['enterprise_size']=='Large').sum(),    p=[0.35,0.35,0.30])

fe['crop_type'] = np.random.choice(
    ['Wheat','Rice','Cotton','Maize','Pulses','Vegetables','Groundnut'],
    size=n, p=[0.18,0.18,0.12,0.14,0.14,0.12,0.12])

fe['irrigation_type'] = np.random.choice(
    ['Rainfed','Canal','Borewell','Sprinkler','Drip'],
    size=n, p=[0.35,0.20,0.20,0.12,0.13])

print('Simulated: landholding_size, crop_type, irrigation_type')

In [ ]:
# Run on full dataset
all_matches = []
for _, row in fe.iterrows():
    for m in match_subsidies(row.to_dict()):
        all_matches.append({'enterprise_id': row['enterprise_id'], **m})

subsidy_match_df = pd.DataFrame(all_matches)
subsidy_match_df.insert(0, 'match_id', [f'SM{i+1:06d}' for i in range(len(subsidy_match_df))])
subsidy_match_df['evaluated_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f'Total matches: {len(subsidy_match_df):,}')
print(f'Avg schemes per enterprise: {len(subsidy_match_df)/len(fe):.1f}')
print()
print(subsidy_match_df['scheme_name'].value_counts().to_string())

In [ ]:
# Fig 7 — Subsidy coverage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Subsidy Matcher Results — SRS §12.5', fontsize=13, fontweight='bold')

ax = axes[0]
sc = subsidy_match_df['scheme_name'].value_counts()
ax.barh([n[:30] for n in sc.index[::-1]], sc.values[::-1],
        color=plt.cm.Blues(np.linspace(0.4, 0.9, len(sc))))
ax.set_xlabel('Eligible Enterprises')
ax.set_title('Coverage per Scheme')

ax = axes[1]
spe = subsidy_match_df.groupby('enterprise_id')['scheme_id'].count()
ax.hist(spe.values, bins=range(0,9), color='#3498db', edgecolor='white', alpha=0.85)
ax.set_xlabel('Number of Eligible Schemes')
ax.set_ylabel('Number of Enterprises')
ax.set_title(f'Schemes per Enterprise (avg = {spe.mean():.1f})')

plt.tight_layout()
plt.savefig('outputs/fig7_subsidy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Demo
for profile in [
    {'enterprise_id':'DEMO-1','enterprise_size':'Small','region':'East',
     'annual_revenue':190,'landholding_size':1.5,'crop_type':'Rice','irrigation_type':'Rainfed'},
    {'enterprise_id':'DEMO-2','enterprise_size':'Large','region':'North',
     'annual_revenue':900,'landholding_size':18,'crop_type':'Wheat','irrigation_type':'Canal'},
]:
    matched = match_subsidies(profile)
    print(f"\n🏡 {profile['enterprise_id']} | {profile['enterprise_size']} | {profile['region']}")
    print(f"   Crop: {profile['crop_type']} | Land: {profile['landholding_size']} acres")
    print(f"   Eligible schemes ({len(matched)}):")
    for m in matched:
        print(f"   ✅ [{m['scheme_id']}] {m['scheme_name']}")
        print(f"      {m['benefits']}")

## 9. Database Table Export
**ER Diagram** — Feature_Vector (Table 5), Credit_Assessment (Table 6), Risk_Explanation (Table 7), Subsidy_Match (Table 9)

In [ ]:
# Feature_Vector table
pd.DataFrame({
    'feature_id':               [f'FV{i+1:05d}' for i in range(len(fe))],
    'farmer_id':                fe['enterprise_id'].values,
    'expense_revenue_ratio':    fe['ERR'].round(4).values,
    'debt_sustainability_ratio':fe['DSR'].round(4).values,
    'net_profit_margin':        fe['NPM'].round(4).values,
    'debt_equity_ratio':        fe['DE_ratio'].round(4).values,
    'climate_resilience_index': fe['CRI'].round(4).values,
    'temperature_stress_indicator': fe['TSI'].round(4).values,
    'market_risk_indicator':    fe['MRI'].round(4).values,
    'encoded_enterprise_size':  fe['enterprise_enc'].values,
    'encoded_region':           fe['region'].values,
    'normalized_vector':        fe[ENGINEERED].round(4).apply(
                                    lambda r: json.dumps(r.to_dict()), axis=1).values,
    'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}).to_csv('outputs/db_feature_vector.csv', index=False)

# Credit_Assessment table
pd.DataFrame({
    'assessment_id':          [f'CA{i+1:05d}' for i in range(len(fe))],
    'farmer_id':              fe['enterprise_id'].values,
    'feature_id':             [f'FV{i+1:05d}' for i in range(len(fe))],
    'probability_of_default': fe['P_default'].values,
    'credit_score':           fe['credit_score'].values,
    'risk_category':          fe['risk_category'].values,
    'model_version':          '1.0-XGB',
    'assessment_date':        datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}).to_csv('outputs/db_credit_assessment.csv', index=False)

# Risk_Explanation table
pd.DataFrame([{
    'explanation_id':              f'EX{i+1:05d}',
    'assessment_id':               f'CA{i+1:05d}',
    'top_risk_increasing_factors': json.dumps(explain(fe.iloc[i].to_dict()).to_dict('records')),
    'top_risk_reducing_factors':   json.dumps([]),
    'shap_values':                 json.dumps({}),
} for i in range(len(fe))]).to_csv('outputs/db_risk_explanation.csv', index=False)

# Subsidy_Match table
subsidy_match_df.to_csv('outputs/db_subsidy_match.csv', index=False)

print('✅ DB tables exported:')
print('   db_feature_vector.csv    (ER Table 5)')
print('   db_credit_assessment.csv (ER Table 6)')
print('   db_risk_explanation.csv  (ER Table 7)')
print('   db_subsidy_match.csv     (ER Table 9)')

## 10. Inference Pipeline & Save Model
**SRS §18.2** — End-to-end function for one new enterprise. Called by FastAPI backend.

In [ ]:
def full_assessment(inp: dict) -> dict:
    """
    Full SRS pipeline for one enterprise.
    Farmer-provided: enterprise_size, region, quarter, annual_revenue,
                     annual_expenses, loan_amount, landholding_size,
                     crop_type, irrigation_type
    From APIs:       avg_temperature, rainfall (IMD)
                     drought_index, flood_risk (NDMA)
                     commodity_price_index, input_cost_index (AGMARKNET)
                     policy_support_score (Govt Policy DB)
    """
    EPS  = 1e-6
    rev  = float(inp.get('annual_revenue', 0)) + EPS
    exp  = float(inp.get('annual_expenses', 0))
    loan = float(inp.get('loan_amount', 0))
    np_  = float(inp.get('net_profit', rev - exp))
    de   = float(inp.get('debt_to_equity', loan / (rev * 2)))
    rain = float(inp.get('rainfall', 200))
    temp = float(inp.get('avg_temperature', 25))
    di   = float(inp.get('drought_index', 0.5))
    fr   = float(inp.get('flood_risk', 0.5))
    comm = float(inp.get('commodity_price_index', 100))
    inpc = float(inp.get('input_cost_index', 100))
    ps   = int(inp.get('policy_support_score', 2))

    rain_norm = np.clip((rain - fe['rainfall'].min()) /
                        (fe['rainfall'].max() - fe['rainfall'].min() + EPS), 0, 1)
    ERR = np.clip(exp/rev, 0, 3)
    NPM = np.clip(np_/rev, -2, 1)
    mri = np.clip(comm/(inpc+EPS), 0.3, 3.0)

    features = {
        'annual_revenue': rev, 'annual_expenses': exp, 'loan_amount': loan,
        'net_profit': np_, 'avg_temperature': temp, 'rainfall': rain,
        'drought_index': di, 'flood_risk': fr,
        'commodity_idx': comm, 'input_cost_idx': inpc, 'policy_score': float(ps),
        'ERR': ERR, 'DSR': np.clip(loan/rev, 0, 5), 'NPM': NPM,
        'DE_ratio': np.clip(de, 0, 5),
        'CRI': np.clip(0.4*rain_norm - 0.35*di - 0.25*fr + 0.35, 0, 1),
        'TSI': np.clip(abs(temp-23.5)/10, 0, 1),
        'MRI': mri,
        'enterprise_enc': float({'Small':0,'Medium':1,'Large':2}.get(inp.get('enterprise_size','Medium'),1)),
        'policy_norm': np.clip((ps-1)/3, 0, 1),
        'quarter_enc': float({'Q1':0,'Q2':1,'Q3':2,'Q4':3}.get(inp.get('quarter','Q1'),0)),
        'financial_stability': np.clip(1-ERR/2, 0, 1),
        'leverage_risk': np.clip(np.clip(de,0,5)/5, 0, 1),
        'repayment_capacity': np.clip(NPM/2+0.5, 0, 1),
    }
    for r in [c.replace('region_','') for c in feature_names if c.startswith('region_')]:
        features[f'region_{r}'] = 1 if inp.get('region','') == r else 0

    X_new  = pd.DataFrame([features]).reindex(columns=feature_names, fill_value=0)
    p_def  = float(xgb.predict_proba(X_new)[0,1])
    score  = round((1 - p_def) * 100, 2)
    risk   = 'Low Risk' if score>=80 else 'Medium Risk' if score>=50 else 'High Risk'

    return {
        'enterprise_id':          inp.get('enterprise_id','NEW'),
        'probability_of_default': round(p_def, 4),
        'credit_score':           score,
        'risk_category':          risk,
        'lending_recommendation': {
            'Low Risk':    'Eligible — proceed with standard loan terms',
            'Medium Risk': 'Conditional — consider reduced amount or additional review',
            'High Risk':   'High risk — flag for manual review',
        }[risk],
        'top_features':     explain(features).to_dict('records'),
        'eligible_subsidies': match_subsidies({
            'enterprise_size': inp.get('enterprise_size',''),
            'region': inp.get('region',''),
            'annual_revenue': rev,
            'landholding_size': float(inp.get('landholding_size', 1.0)),
            'crop_type': inp.get('crop_type',''),
            'irrigation_type': inp.get('irrigation_type',''),
        }),
        'model_version': '1.0-XGB',
        'assessed_at':   datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }

print('✅ Inference pipeline ready.')

In [ ]:
# Test on 2 sample enterprises
for s in [
    {'enterprise_id':'HIGH-RISK','enterprise_size':'Small','region':'East','quarter':'Q3',
     'annual_revenue':250,'annual_expenses':390,'loan_amount':200,'net_profit':-140,
     'debt_to_equity':1.8,'avg_temperature':33,'rainfall':155,'drought_index':0.78,
     'flood_risk':0.65,'commodity_price_index':82,'input_cost_index':115,
     'policy_support_score':1,'landholding_size':1.2,'crop_type':'Pulses','irrigation_type':'Rainfed'},
    {'enterprise_id':'LOW-RISK','enterprise_size':'Large','region':'North','quarter':'Q1',
     'annual_revenue':950,'annual_expenses':520,'loan_amount':100,'net_profit':430,
     'debt_to_equity':0.25,'avg_temperature':22,'rainfall':230,'drought_index':0.18,
     'flood_risk':0.12,'commodity_price_index':132,'input_cost_index':92,
     'policy_support_score':4,'landholding_size':18,'crop_type':'Wheat','irrigation_type':'Canal'},
]:
    r = full_assessment(s)
    emoji = '🟢' if r['credit_score']>=80 else '🟡' if r['credit_score']>=50 else '🔴'
    print(f"\n{emoji} {r['enterprise_id']}")
    print(f"   Score: {r['credit_score']}/100 | {r['risk_category']} | P(Default): {r['probability_of_default']}")
    print(f"   {r['lending_recommendation']}")
    print(f"   Eligible subsidies: {[m['scheme_name'] for m in r['eligible_subsidies']]}")

In [ ]:
# Save model
with open('outputs/xgboost_model.pkl','wb') as f:
    pickle.dump({
        'model': xgb, 'feature_names': feature_names,
        'engineered_features': ENGINEERED,
        'version': '1.0-XGB',
        'roc_auc': round(roc_auc_score(y_test, xgb_prob), 4),
        'f1':      round(f1_score(y_test, xgb_pred), 4),
    }, f)

fe.to_csv('outputs/scored_dataset.csv', index=False)

print('=' * 50)
print(' SUMMARY')
print('=' * 50)
print(f' Dataset    : {len(fe):,} enterprises')
print(f' Final model: XGBoost')
print(f' ROC-AUC    : {roc_auc_score(y_test, xgb_prob):.4f}')
print(f' F1 Score   : {f1_score(y_test, xgb_pred):.4f}')
for cat in CATS:
    n = risk_dist.get(cat, 0)
    print(f' {cat:<15}: {n:,} ({n/len(fe):.1%})')
print('\n✅ All outputs saved to /outputs/')